# Compare on the benchmark

In this notebook, we walk through running the script you wrapped in Run your own script on a few single-coefficient estimation (SCE) problem instances, and comparing its errors with those of already-benchmarked algorithms. The steps are:

1. Open the SCE collection from the benchmark fetched in the [Install and fetch the benchmark](install_and_fetch.ipynb) notebook.
2. Import the custom regression algorithm we created in the [Run your own script](run_your_own_script.ipynb) notebook.
3. Run the benchmark's sweep on three instances.
4. Read the statuses of the trials.
5. Divide the errors by the baseline.
6. Plot profile curves.

This notebook uses `my_regression.py` and `my_regression_script.py` from Run your own script, keep them in the same directory as this notebook.

## Opening the SCE collection

First download the benchmark as in Install and fetch the benchmark. If it is already downloaded in `dp-benchmark/official-release/`, the cell downloads nothing.

In [ ]:
from pathlib import Path

from dpstats import fetch_benchmark

workspace = fetch_benchmark(Path("dp-benchmark"))

Then we open the [SCE](https://saminulh.github.io/dpstats/single_coefficient_estimation/) collection with [open_collection](https://saminulh.github.io/dpstats/library/reference/collections/dpstats.open_collection/). This manages all the algorithms, instances, and already-computed results for this task. It reads the official release as well as any records you may already have in `dp-benchmark/local`. The cell prints the number of instances and configured algorithms in the release.

In [ ]:
from dpstats import open_collection
from dpstats.tasks import SingleCoefficientEstimation

collection = open_collection(workspace, SingleCoefficientEstimation)
instances = collection.load_instances(source="release")
released = collection.load_algorithms(source="release")
print(f"SCE release: {len(instances)} instances, {len(released)} algorithms")

## Configuring our algorithm

The algorithm `MyRegression` we implemented in [Run your own script](run_your_own_script.ipynb) is for the [Linear Regression](https://saminulh.github.io/dpstats/linear_regression/) task, which asks for the whole OLS coefficient, while SCE asks for just a single coefficient. We can use the [ToLinearRegression](https://saminulh.github.io/dpstats/single_coefficient_estimation/transformations/ToLinearRegression/) transformation to turn the SCE instance into a linear regression one that `MyRegression` can accept. The next cell constructs the algorithm for SCE and prints its UUID.

In [ ]:
from dpstats.algorithm import CompositeAlgorithm
from dpstats.tasks.single_coefficient_estimation.transformations import ToLinearRegression
from my_regression import MyRegression

my_alg = CompositeAlgorithm(
    transformations=(ToLinearRegression(),),
    algorithm=MyRegression(),
    label="MyRegression for SCE",
)
print(my_alg)
print(my_alg.uuid)

To run on instances without public bounds, we can also put [ApproxBounds](https://saminulh.github.io/dpstats/linear_regression/transformations/ApproxBounds/) in front of `MyRegression`. It spends part of the privacy budget to estimate the missing bounds and clips the data to them, so `MyRegression` gets the finite bounds it needs. [Transformations and composition](extend_with_transformations.ipynb) shows more transformations and how they compose.

In [ ]:
from dpstats.tasks.linear_regression.transformations import ApproxBounds

my_alg_bounded = CompositeAlgorithm(
    transformations=(ToLinearRegression(), ApproxBounds()),
    algorithm=MyRegression(),
    label="MyRegression with ApproxBounds for SCE",
)
print(my_alg_bounded)
print(my_alg_bounded.uuid)

## Running the benchmark's sweep

The protocol we use for benchmarking the algorithms is saved in the official release as `current.json`. We can access it via `collection.load_sweep("current")`. It contains the budgets, seeds, and problem instances we run every configured algorithm on. You can override any of these parameters by passing them to `get_sweep_status`, `run_sweep` and `load_results`. We override algorithms to run only our two configurations and we choose a few instances to run. If you wish to sweep over all instances in the public corpus, you can omit the `instances` argument.

This cell prints the budget and seeds in the sweep as well as how many of the sweep trials have already been completed.

In [ ]:
collection.add_algorithm(my_alg, registry="mine")
collection.add_algorithm(my_alg_bounded, registry="mine")
sweep = collection.load_sweep("current")
print("budgets:", sweep["budgets"])
print("seeds:", sweep["seeds"])

instance_uuids = [
    "i5-6JXYHSJJFKZFVZQZ",  # dataverse_70000250_reg_4:single_coefficient_estimation/coefficient_3
    "i5-KBZJOSRQCCIWL22E",  # icpsr_194887_reg_4__2:single_coefficient_estimation/coefficient_1
    "i5-4AQGPI4IZOPM636F",  # dataverse_70000009_reg_1:single_coefficient_estimation/coefficient_1
]

selection = {"algorithms": [my_alg, my_alg_bounded], "instances": instance_uuids}
print(collection.get_sweep_status(sweep, **selection)["cells"])

The third instance has no public bound on one of its features, so `MyRegression` refuses it. The benchmark counts a refusal as a failure, so its curve below cannot rise above two thirds. `MyRegression` with ApproxBounds estimates that bound privately and runs on all three.

We can now call [run_sweep](https://saminulh.github.io/dpstats/library/reference/collections/dpstats.TaskCollection/#dpstats.TaskCollection.run_sweep) to compute all of the results. Two configurations, three instances, four budgets and ten seeds make 240 trials. The cell prints the number of trials and how many ended with each status. The results are saved in `dp-benchmark/local` and can be resumed if the sweep was interrupted.

In [ ]:
print("before:", collection.get_sweep_status(sweep, **selection)["cells"])
results = collection.run_sweep(sweep, time_limit=None, memory_limit_gb=None, **selection)
print("after:", collection.get_sweep_status(sweep, **selection)["cells"])
statuses = results["status"].map(lambda status: status.value).value_counts().to_dict()
print(f"{len(results)} trials, statuses: {statuses}")

## Reading the statuses

Each trial ends with one status:

1. `success`: the algorithm released an output.
2. `invalid_regime`: the algorithm refused the problem from its public information.
3. `no_release`: the algorithm spent its budget without releasing an output.
4. `timeout` and `memory_limit_exceeded`: the trial ran out of time or memory.
5. `algorithm_error`: the algorithm raised an error.

The next cell gives the share of each status for your configurations with [summarize_outcomes](https://saminulh.github.io/dpstats/library/reference/analysis/dpstats.analysis.summarize_outcomes/). Each share is out of every trial of that configuration. Read this table before the errors below.

In [ ]:
from dpstats.analysis import summarize_outcomes

shares = summarize_outcomes(results, by=["algorithm_label"])
print(shares.pivot(index="algorithm_label", columns="status", values="share").to_string())

## Computing metrics and baselines

The main metric for SCE is [absolute_error_to_ols](https://saminulh.github.io/dpstats/single_coefficient_estimation/metrics/#absolute-error-to-ols), $|\hat{\beta}_j - \hat{\beta}_{{\rm OLS}, j}|$: the distance from the estimate to the nonprivate ordinary least squares coefficient. The corresponding baseline is `published_se`, the standard error that the source study published for this coefficient, see [Methodology](https://saminulh.github.io/dpstats/methodology/) for more details. We normalize the metric by the baseline, so that the error is expressed in units of standard error, which makes it easier to compare across instances. For comparison, we also add two configurations whose results are included in the downloaded release: AdaSSP, scaled and SSP, scaled, which estimate bounds privately and standardize the data before running AdaSSP and SSP.

In [ ]:
from dpstats.tasks import list_built_in_metrics
from dpstats.analysis import measure_results, compute_error_ratio

by_label = {algorithm.label: algorithm for algorithm in released.values()}
adassp = by_label["AdaSSP, scaled"]
ssp = by_label["SSP, scaled"]
results = collection.load_results(sweep, instances=instance_uuids, algorithms=[my_alg, my_alg_bounded, adassp, ssp])

metric = list_built_in_metrics(SingleCoefficientEstimation)["absolute_error_to_ols"]
print(f"metric: {metric.name}")
measurements = measure_results(results, metrics=[metric])
ratios = compute_error_ratio(measurements, "published_se", metric=metric)
ratios["eps"] = ratios["budget"].map(lambda budget: budget.eps)
table = ratios.pivot_table(
    index=["algorithm_label", "instance_label"], columns="eps", values=metric.name, aggfunc="median"
)
print(table.round(2).to_string())

## Plotting profile curves

At each threshold $\tau$, a profile curve shows the share of trials whose ratio is at most $\tau$, averaged over the three instances. We plot your configurations beside AdaSSP, scaled and SSP, scaled, built on [AdaSSP](https://saminulh.github.io/dpstats/linear_regression/algorithms/AdaSSP/) and [SSP](https://saminulh.github.io/dpstats/linear_regression/algorithms/SSP/), at $\varepsilon = 1$. The next cell computes the curves with [compute_profile_curves](https://saminulh.github.io/dpstats/library/reference/analysis/dpstats.analysis.compute_profile_curves/) and plots them with [plot_profiles](https://saminulh.github.io/dpstats/library/reference/analysis/dpstats.analysis.plot_profiles/).

In [ ]:
import numpy as np

from dpstats.analysis import compute_profile_curves, plot_profiles

eps = 1.0

ratios_at_eps = ratios[
    ratios["budget"].map(lambda budget: budget.eps) == eps
]

curves = compute_profile_curves(ratios_at_eps, np.linspace(1.0, 10.0, 20), metric=metric)
figure = plot_profiles(
    curves,
    xlabel="absolute error / published_se",
    ylabel="share of trials within τ",
    xscale="log",
)
figure.show()

## Next steps

To extend the comparison, change the instances or the configuration, and check `get_sweep_status` before running more of the sweep. Keep the sweep from `load_sweep("current")` and the `published_se` baseline, so that your results stay comparable with the released ones.

The [analysis reference](https://saminulh.github.io/dpstats/library/reference/analysis/) lists more summaries and plots, and the [Metrics and baselines notebook](metrics_and_baselines.ipynb) explains metrics and baselines in more detail. To submit your algorithm to the benchmark, follow [Submit](https://saminulh.github.io/dpstats/submit/submission/).